# Lab 3 — Index Business Graph Images into ChromaDB

This notebook embeds each business graph images using **Cohere embed-v4.0** and stores the vectors in a local **ChromaDB** collection to illustrate embed-v4 multimodal capabilities.  The dataset comes from reddit, r/dataisbeautiful and was manually curated.

**Before running:** make sure you have copied `.env.example` to `.env` at the project root and update.

In [8]:
import os
import shutil
import time
import base64
from dotenv import load_dotenv
from tqdm.notebook import tqdm
import cohere
from cohere.errors import TooManyRequestsError
import chromadb
from PIL import Image

load_dotenv(dotenv_path='../.env', override=True)

chromadb_path = os.environ['CHROMA_DB_PATH']
collection_name = 'Business_Graphs'
MODEL = os.environ['EMBED_MODEL']
images_folder = './dataset/'
co = cohere.ClientV2(
    base_url=os.environ['EMBED_BASE_URL'],
    api_key=os.environ['EMBED_API_KEY'],
    timeout=300)

# ── Fresh start: wipe chroma_db folder and recreate ───────────────────────────
if os.path.exists(chromadb_path):
    shutil.rmtree(chromadb_path)
    print(f"Deleted existing ChromaDB at '{chromadb_path}'")

chroma_client = chromadb.PersistentClient(path=chromadb_path)
collection = chroma_client.create_collection(name=collection_name)
print(f"Created fresh collection '{collection_name}'")

# ── Helpers ───────────────────────────────────────────────────────────────────
def local_image_to_base64(filename):
    try:
        with open(filename, "rb") as image_file:
            image_data = image_file.read()
            return base64.b64encode(image_data).decode('utf-8'), len(image_data)
    except FileNotFoundError:
        print(f"===> ERROR: File '{filename}' not found.")
        return None, None
    except Exception as e:
        print(f"===> ERROR: {str(e)}")
        return None, None

MAX_RETRIES = 5

def embed_with_retry(images):
    for attempt in range(MAX_RETRIES):
        try:
            return co.embed(
                images=images,
                model=MODEL,
                input_type='search_document',
                embedding_types=['float']
            ).embeddings.float_[0]
        except TooManyRequestsError as e:
            if attempt == MAX_RETRIES - 1:
                raise
            retry_after_ms = int(e.headers.get('retry-after-ms', 60000))
            wait_s = retry_after_ms / 1000 + 1
            print(f"  Rate limit hit — waiting {wait_s:.1f}s (attempt {attempt + 1}/{MAX_RETRIES})...")
            time.sleep(wait_s)

# ── Index all images ──────────────────────────────────────────────────────────
jpg_files = sorted(
    [f for f in os.listdir(images_folder) if f.lower().endswith(('.jpg', '.jpeg'))],
    key=lambda x: x.lower()
)

for index, filename in enumerate(tqdm(jpg_files, desc="Indexing images")):
    file_path = os.path.join(images_folder, filename)
    base64_encoded_image, image_size = local_image_to_base64(file_path)

    if base64_encoded_image is None:
        continue

    print(f'===> index={index} filename={filename}')
    embeddings = embed_with_retry(
        images=[f"data:image/jpg;base64,{base64_encoded_image}"]
    )

    collection.add(
        documents=[filename],
        embeddings=[embeddings],
        ids=[str(index + 1)],
        metadatas=[{
            'type': 'Graph',
            'filename': filename,
            'image_size': image_size,
            'index': index + 1,
        }]
    )
    time.sleep(2)

print(f"\nDone. Indexed {collection.count()} images into '{collection_name}'.")


Deleted existing ChromaDB at './chroma_db'


UniqueConstraintError: Collection Business_Graphs already exists

# Chrome db collection stats 
Get collection stats after the above indexing job.

In [ ]:
# ── ChromaDB sanity check ──────────────────────────────────────────────────

total = collection.count()
print(f"Total documents indexed: {total}\n")

# Peek at 5 sample rows
sample = collection.peek(limit=5)

print(f"{'ID':<6} {'Document':<25} {'Metadata'}")
print("-" * 70)
for doc_id, doc, meta in zip(sample["ids"], sample["documents"], sample["metadatas"]):
    print(f"{doc_id:<6} {doc:<25} {meta}")

# Search query example 1.
Text search to best semantic matching business graphs

In [ ]:
import os
import base64
from dotenv import load_dotenv
import cohere
import chromadb
from PIL import Image

load_dotenv(dotenv_path='../.env', override=True)

chromadb_path = os.environ['CHROMA_DB_PATH']
chroma_client = chromadb.PersistentClient(path=chromadb_path)
collection_name = 'Business_Graphs'
collection = chroma_client.get_collection(name=collection_name)
MODEL = os.environ['EMBED_MODEL']
images_folder = './dataset/'

co = cohere.ClientV2(
    base_url=os.environ['EMBED_BASE_URL'],
    api_key=os.environ['EMBED_API_KEY'], 
    timeout=300)

def search(query: str = None, image_path: str = None, top_k: int = 5):
    """Search ChromaDB by text query, local image path, or both combined.

    Modes
    -----
    text only  : pass query, leave image_path=None
    image only : pass image_path, leave query=None
    text+image : pass both — the embedding fuses them into a single vector
    """
    import mimetypes

    if query is None and image_path is None:
        raise ValueError("Provide at least one of 'query' (text) or 'image_path'.")

    image_data_uri = None
    if image_path is not None:
        with open(image_path, "rb") as f:
            image_bytes = f.read()
        b64 = base64.b64encode(image_bytes).decode("utf-8")
        content_type = mimetypes.guess_type(image_path)[0] or "image/jpeg"
        image_data_uri = f"data:{content_type};base64,{b64}"

    if query is not None and image_path is not None:
        # ── Text + Image (fused) ───────────────────────────────────────────────
        print('***Results based on both search text and input image\n')
        result = co.embed(
            inputs=[{
                "content": [
                    {"type": "text", "text": query},
                    {"type": "image_url", "image_url": {"url": image_data_uri}}
                ]
            }],
            model=MODEL,
            input_type='search_query',
            embedding_types=['float']
        )
       

    elif query is None and image_path is not None:
        # ── Image-only ─────────────────────────────────────────────────────────
        print('*** Results based on input image only\n')
        result = co.embed(
            inputs=[{
                "content": [
                    {"type": "image_url", "image_url": {"url": image_data_uri}}
                ]
            }],
            model=MODEL,
            input_type='image',
            embedding_types=['float']
        )

    else:
        # ── Text-only ──────────────────────────────────────────────────────────
        print('*** Results based on text query only\n')
        result = co.embed(
            texts=[query],
            model=MODEL,
            input_type='search_query',
            embedding_types=['float']
        )



    query_embedding = result.embeddings.float_[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=['metadatas', 'distances']
    )

    matches = []
    for metadata, distance in zip(results['metadatas'][0], results['distances'][0]):
        matches.append({
            'filename': metadata['filename'],
            'similarity': round(1 - distance, 4),
            'filepath': os.path.join(images_folder, metadata['filename'])
        })
    return matches

# --- Try a search query ---
query = "show me bar chart on coffee"
top_k = 3

print(f"*** Search query: '{query}'\n")

matches = search(query, top_k=top_k)
print(f"*** Top {top_k} semantic search results:\n")
for i, match in enumerate(matches):
    print(f"[{i+1}] {match['filename']}  (similarity: {match['similarity']})")
    img = Image.open(match['filepath'])
    img.thumbnail((400, 400))
    display(img)
    print()

# Search query example 2. 
Image search — find visually similar business graphs

In [ ]:
# ── Search by image only ──────────────────────────────────────────────────────
# Update this path to the local image you want to search with
query_image_path = './search_images/g1_microsoft_june_2022.jpg'
top_k = 3

print(f"*** Input image {query_image_path} for search:")
query_img = Image.open(query_image_path)
query_img.thumbnail((400, 400))
display(query_img)
print()

matches = search(image_path=query_image_path, top_k=top_k)

print(f"*** Top {top_k} semantic search results:\n")
for i, match in enumerate(matches):
    print(f"[{i+1}] {match['filename']}  (similarity: {match['similarity']})")
    result_img = Image.open(match['filepath'])
    result_img.thumbnail((400, 400))
    display(result_img)
    print()

# Search query example 3 
Combined image + text search — fused multimodal query

In [ ]:
# ── Search by image + text (fused multimodal query) ──────────────────────────
# Update the path and text query below
query_image_path = './search_images/g1_microsoft_june_2022.jpg'
query_text = "similar plots comparing before and after IPO"
top_k = 3

print(f"** Search query: '{query_text}' + input image {query_image_path}:")
query_img = Image.open(query_image_path)
query_img.thumbnail((400, 400))
display(query_img)

matches = search(query=query_text, image_path=query_image_path, top_k=top_k)

print(f"*** Top {top_k} semantic results:\n")
for i, match in enumerate(matches):
    print(f"[{i+1}] {match['filename']}  (similarity: {match['similarity']})")
    result_img = Image.open(match['filepath'])
    result_img.thumbnail((400, 400))
    display(result_img)
    print()